# LAB 03 — Statistics & Exploratory Data Analysis

**Module:** Statistics & Exploratory Data Analysis
**Session:** 03
**Duration:** 150 minutes (10-minute break included)
**Audience:** Başlangıç seviyesi Python / Machine Learning katılımcıları (LAB02'yi tamamlamış olmalı)

## LAB02'den Devam

Bu LAB, **LAB02 — Python for Machine Learning & Data Manipulation**'ın devamıdır. LAB02'de öğrendiğiniz
NumPy, Pandas (`Series`/`DataFrame`), data loading, filtering/sorting ve groupby/aggregation konuları
burada **sıfırdan tekrar öğretilmeyecek** — bu bilgiler, aşağıdaki istatistiksel analiz araçlarını
uygulamak için doğrudan kullanılacaktır.

## Bu LAB'ın Ana Hedefi

Bir veri kümesini bir Machine Learning modeline vermeden önce **istatistiksel olarak incelemek**, veri
kalitesi sorunlarını belirlemek ve bulguları teknik olarak yorumlamak.

## Bu LAB'ın Sonunda Yapabilecekleriniz

1. Exploratory Data Analysis (EDA)'nın ne olduğunu ve neden önemli olduğunu açıklamak.
2. Descriptive statistics (mean, median, mode) hesaplamak ve yorumlamak.
3. Dispersion ölçülerini (variance, standard deviation, IQR) hesaplamak.
4. IQR yöntemiyle potansiyel outlier'ları belirlemek.
5. Bir değişkenin dağılım şeklini (skewness) değerlendirmek.
6. Veri kalitesi değerlendirmesi (data quality assessment) yapmak.
7. Correlation matrix oluşturmak ve doğru yorumlamak.
8. Kategorik gruplar arasında istatistiksel karşılaştırma yapmak.
9. Matplotlib ile EDA'ya özgü grafikler (histogram, boxplot, scatter) oluşturmak.

## Kullanılan Veri Kümesi

Bu LAB'da **tek bir veri kümesi** kullanılır: **E-Commerce Shipping Data**
(`data/ecommerce_shipping/Train.csv`) — LAB02'de kullandığınız aynı dataset. Northwind veri kümesi bu
LAB'da **kullanılmaz** (Northwind yalnızca LAB02'nin Merge & Join bölümüne özgüdür).

> Bu veri kümesi, istatistiksel yöntemleri öğretmek için kullanılan **temsili bir eğitim kaynağıdır**;
> gerçek bir lojistik şirketinin performansı hakkında kesin bir değerlendirme olarak ele alınmamalıdır.

> **Not:** Bu LAB'da **model eğitimi, preprocessing pipeline, imputation, encoding veya scaling**
> yapılmaz. Bu işlemler **Module 4 — Data Preparation** kapsamında ele alınacaktır. Burada amaç, veriyi
> modele vermeden önce *anlamaktır*.


---
## Section 1 — LAB02 Review & EDA Introduction
**Süre: 10 dakika**

### Learning Objective

Exploratory Data Analysis (EDA)'nın ne olduğunu ve bir Machine Learning projesindeki rolünü anlamak;
LAB02'deki dataset'i tekrar yükleyerek hızlı bir hatırlatma yapmak.

### Concept Explanation

- **Exploratory Data Analysis (EDA)**: Bir veri kümesini modellemeden önce sayılar, özetler ve basit
  grafiklerle tanıma sürecidir. Amaç, veriyi "hissetmek" ve varsayımları erken test etmektir.
- **Why EDA Matters in Machine Learning**: Bir model, verideki hataları, eksiklikleri veya aşırı
  çarpıklıkları "fark etmez" — onları olduğu gibi öğrenir. EDA, bu sorunları modele vermeden önce
  yakalamanızı sağlar.
- **Data Understanding**: Her kolonun ne anlama geldiğini, hangi değer aralığında olabileceğini ve hangi
  veri tipinde tutulduğunu bilmek.
- **Statistical Thinking**: Tek bir sayıya (örn. ortalama) bakmak yerine, bir değişkenin merkezini,
  yayılımını ve şeklini birlikte değerlendirmek.
- **EDA vs Data Preparation**: EDA, veriyi **gözlemler ve raporlar**; Data Preparation (Module 4) ise
  eksik değerleri doldurma, kategorik değişkenleri kodlama gibi veriyi **değiştiren** işlemleri yapar. Bu
  LAB'da yalnızca gözlem ve raporlama yapılacaktır.
- **Descriptive vs Inferential Statistics**: Descriptive statistics, elimizdeki veriyi özetler (örn. "bu
  10999 gönderinin ortalama maliyeti X'tir"). Inferential statistics, bir örneklemden daha büyük bir
  popülasyon hakkında çıkarım yapmaya çalışır (örn. hipotez testleri, güven aralıkları). Bu LAB
  ağırlıklı olarak descriptive statistics'e odaklanır; `Advanced Challenge 1`'de inferential bir araç
  olan **bootstrap confidence interval**'a kısaca değinilecektir.

### Hands-On


In [ ]:
# Same environment setup pattern as LAB02: resolve the project root, add src/ to
# sys.path, then load the shared configuration and the dataset.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


In [ ]:
from data_loading import load_config, load_ecommerce_shipping

config = load_config()
df = load_ecommerce_shipping(config)
print("Dataset shape:", df.shape)


**Hızlı hatırlatma (LAB02):** `shape`, `columns`, `dtypes` ile dataset'in boyutunu ve yapısını kontrol
etme adımları.


In [ ]:
print("Shape (rows, columns):", df.shape)
print("Columns:", df.columns.tolist())
df.dtypes


In [ ]:
# Separate numerical and categorical columns (pandas 3.x stores text columns as "string" dtype).
numerical_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(include="string").columns.tolist()

print("Numerical columns  :", numerical_columns)
print("Categorical columns:", categorical_columns)


### Target Distribution

Hatırlatma: `Reached.on.Time_Y.N` sütununda **`1 = Late`**, **`0 = On Time`** anlamına gelir (Kaggle
açıklaması, LAB02'de ele alınmıştı). Bu LAB boyunca "Late Delivery Rate" hesaplarken bu tanımı
kullanacağız.


In [ ]:
target_counts = df["Reached.on.Time_Y.N"].value_counts().sort_index()
target_pct = df["Reached.on.Time_Y.N"].value_counts(normalize=True).sort_index() * 100

print("Target value counts (0=On Time, 1=Late):")
print(target_counts)
print()
print("Target distribution (%):")
print(target_pct.round(2))


### Output Interpretation

- Dataset, LAB02'de incelediğimiz **10999 satır, 12 sütun**luk aynı şekle sahiptir (`Train.csv` dosyası
  değişmemiştir).
- `ID` sütunu, her gönderiyi tekil olarak tanımlayan bir **observation identifier**'dır — bir ölçüm
  değeri değildir, bu yüzden istatistiksel hesaplamalara (mean, correlation vb.) dahil edilmeyecektir.
- `Customer_rating` ve `Prior_purchases` sayısal (`int64`) olarak saklanır, ancak **domain anlamları**
  tartışmaya açıktır: `Customer_rating` 1-5 arası ordinal bir puan, `Prior_purchases` ise bir sayım
  (count) değişkenidir. Bu ayrım, Section 2'de ortalamanın ne kadar anlamlı olduğunu tartışırken önemli
  olacaktır.
- Target dağılımı, gönderilerin çoğunluğunun (`~%60`) geç (`Late`) olarak işaretlendiğini gösterir — bu,
  dataset'in dengesiz (imbalanced) olmadığını, aksine "late" sınıfının biraz daha fazla temsil edildiğini
  gösterir.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Using categorical_columns, print the number of unique values (nunique()) for each column.
# TODO: Decide (in a short comment) whether "Customer_rating" should be treated as numerical or
#       categorical/ordinal for EDA purposes, and justify your answer in one sentence.

# for col in categorical_columns:
#     print(col, df[col].nunique())


---
## Section 2 — Descriptive Statistics
**Süre: 20 dakika**

### Learning Objective

Mean, median ve mode hesaplamalarını gerçekleştirmek; bu üç ölçünün ne zaman farklı davrandığını ve
hangisinin hangi durumda daha anlamlı olduğunu yorumlamak.

### Concept Explanation

- **Mean (Aritmetik Ortalama)**: Tüm değerlerin toplamının gözlem sayısına bölünmesidir. Her değerden
  etkilenir — özellikle **aşırı değerlerden (outlier)** kolayca etkilenir.
- **Median (Medyan)**: Veriyi sıraladıktan sonra ortadaki değerdir. Aşırı değerlerden mean'den çok daha
  az etkilenir (bu yüzden **robust** bir ölçü olarak adlandırılır).
- **Mode (Mod)**: En sık tekrar eden değerdir. Özellikle kategorik veya az sayıda farklı değer alan
  sayısal (ordinal) değişkenler için anlamlıdır.
- **Measures of Central Tendency**: Mean, median ve mode, bir değişkenin "tipik" veya "merkezi"
  değerinin ne olduğunu farklı açılardan özetleyen ölçülerdir.

### Mathematical Explanation

Mean için:

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

Median, sıralanmış veride ortadaki değer (n tek ise tam ortanca, n çift ise ortadaki iki değerin
ortalaması) olarak tanımlanır; mode ise en yüksek frekansa sahip değerdir — bu ikisi için kapalı bir
formülden çok bir **prosedür** tanımlanır.

### Hands-On

Bu bölümde dört sayısal değişken kullanılacaktır: `Cost_of_the_Product`, `Weight_in_gms`,
`Customer_rating`, `Prior_purchases`.


In [ ]:
numeric_features = ["Cost_of_the_Product", "Weight_in_gms", "Customer_rating", "Prior_purchases"]

central_tendency = pd.DataFrame(
    {
        "mean": df[numeric_features].mean(),
        "median": df[numeric_features].median(),
        "mode": [df[col].mode().iloc[0] for col in numeric_features],
    }
)
central_tendency


### NumPy vs Pandas

Mean ve median için NumPy ve Pandas aynı sonucu verir; ancak NumPy'da hazır bir `mode` fonksiyonu
**yoktur** — bu, Pandas'ın `.mode()` metodunun pratik bir avantajıdır.


In [ ]:
cost = df["Cost_of_the_Product"].to_numpy()

print("Pandas mean  :", df["Cost_of_the_Product"].mean())
print("NumPy mean   :", np.mean(cost))
print()
print("Pandas median:", df["Cost_of_the_Product"].median())
print("NumPy median :", np.median(cost))
print()
print("NumPy has no built-in mode function; Pandas Series.mode() is the practical choice.")


### Analysis

Aşağıdaki tabloda (`central_tendency`) gözlemlenebilecek bulgular:

- **`Cost_of_the_Product`**: Mean (~210.2), Median'den (214.0) biraz **küçüktür**. Fark küçük olsa da,
  dağılımın hafifçe sola (negatif) çarpık olabileceğine işaret eder — bunu Section 4'te `skew()` ile
  doğrulayacağız.
- **`Weight_in_gms`**: Mean (~3634), Median'den (4149) **belirgin şekilde küçüktür**. Bu, dağılımın tek
  tepe noktalı (unimodal) ve simetrik olmadığının bir işaretidir.
- **`Customer_rating`**: Mean (~2.99), Median (3) ve Mode (3) **birbirine çok yakındır** — bu, 1-5
  arasındaki puanların veri kümesinde oldukça simetrik dağıldığını gösterir.
- **`Prior_purchases`**: Mean (~3.57), Median'den (3.0) **belirgin şekilde büyüktür**. Bu, bazı
  müşterilerin çok yüksek "prior purchases" değerlerine sahip olup ortalamayı yukarı çektiğini gösterir
  — **outlier'ların mean üzerindeki etkisinin** gerçek bir örneğidir. Median, bu etkiden korunduğu için
  burada "tipik müşteri" için daha güvenilir bir özet sunar.

**Ordinal değişkenler hakkında not:** `Customer_rating` gibi ordinal (sıralı kategori) bir değişken için
mean hesaplamak yaygın bir pratiktir, ancak tartışmalıdır — "3.5 yıldız" gibi bir ortalama, aradaki
aralıkların eşit olduğunu *varsayar*. Mode ve median, ordinal veriler için genellikle daha savunulabilir
ölçülerdir.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Compute mean, median, and mode for "Discount_offered" and "Customer_care_calls".
# TODO: For each variable, state in a comment whether mean or median seems more representative.

# for col in ["Discount_offered", "Customer_care_calls"]:
#     print(col, df[col].mean(), df[col].median(), df[col].mode().iloc[0])


---
## Section 3 — Dispersion, Quantiles & Outliers
**Süre: 25 dakika**

### Learning Objective

Variance, standard deviation, quantile ve IQR hesaplamalarını gerçekleştirmek; IQR yöntemiyle potansiyel
outlier'ları belirlemek.

### Concept Explanation

- **Variance**: Değerlerin ortalamadan ne kadar uzaklaştığının karesinin ortalamasıdır; birimi orijinal
  değişkenin biriminin karesidir (örn. gram²), bu da doğrudan yorumlanmasını zorlaştırır.
- **Standard Deviation**: Variance'ın kareköküdür; orijinal değişkenle **aynı birimde** olduğu için
  genellikle daha yorumlanabilirdir.
- **Range**: Maksimum ile minimum arasındaki farktır; tek bir aşırı değerden kolayca etkilenir.
- **Quantiles / Percentiles / Quartiles**: Veriyi eşit parçalara bölen sınır değerlerdir. Quartile'lar
  veriyi dörde böler: Q1 (%25), Q2 (%50, medyan), Q3 (%75).
- **Interquartile Range (IQR)**: Q3 - Q1; verinin "orta yarısının" ne kadar yayıldığını gösterir, uç
  değerlerden Variance/Std'den daha az etkilenir.
- **Outlier Detection**: IQR tabanlı yöntem, Q1 ve Q3'ten 1.5×IQR kadar uzaktaki değerleri potansiyel
  outlier olarak işaretler.
- **Robust Statistics**: Median ve IQR gibi, aşırı değerlerden az etkilenen istatistiklere "robust"
  denir — Mean ve Variance ise bu anlamda robust değildir.

### Mathematical Explanation

Population variance ve standard deviation:

$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2 \qquad \sigma = \sqrt{\sigma^2}$$

Sample variance ve standard deviation (Bessel's correction, `ddof=1`):

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2 \qquad s = \sqrt{s^2}$$

IQR ve outlier sınırları:

$$IQR = Q_3 - Q_1$$

$$\text{Lower Bound} = Q_1 - 1.5 \times IQR \qquad \text{Upper Bound} = Q_3 + 1.5 \times IQR$$

### Hands-On


In [ ]:
dispersion_features = ["Weight_in_gms", "Cost_of_the_Product"]

for col in dispersion_features:
    print(f"--- {col} ---")
    print("min             :", df[col].min())
    print("max             :", df[col].max())
    print("range           :", df[col].max() - df[col].min())
    print("variance (ddof=1):", df[col].var())
    print("std      (ddof=1):", df[col].std())
    print()


### Sample vs Population Variance — `ddof` Parametresi

Pandas, `.var()` ve `.std()` için varsayılan olarak **sample variance** kullanır (`ddof=1`, paydada
`n-1`). NumPy ise varsayılan olarak **population variance** kullanır (`ddof=0`, paydada `n`). Büyük
`n` değerlerinde (bu dataset'te `n=10999`) fark genellikle çok küçüktür, ancak iki kütüphane
karşılaştırılırken bu varsayılan farkı bilmek önemlidir.


In [ ]:
weight = df["Weight_in_gms"].to_numpy()

print("Pandas .var() [ddof=1, default] :", df["Weight_in_gms"].var())
print("NumPy  .var(ddof=1)              :", np.var(weight, ddof=1))
print("NumPy  .var(ddof=0) [default]    :", np.var(weight, ddof=0))
print()
print("Pandas .std() [ddof=1, default] :", df["Weight_in_gms"].std())
print("NumPy  .std(ddof=0) [default]    :", np.std(weight, ddof=0))


### Quantiles & IQR


In [ ]:
for col in dispersion_features:
    q1 = df[col].quantile(0.25)
    q2 = df[col].quantile(0.50)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    print(f"--- {col} ---")
    print(f"Q1 (25%): {q1}")
    print(f"Q2 (50%, median): {q2}")
    print(f"Q3 (75%): {q3}")
    print(f"IQR: {iqr}")
    print()


### IQR Method — Worked Example on `Weight_in_gms`


In [ ]:
q1 = df["Weight_in_gms"].quantile(0.25)
q3 = df["Weight_in_gms"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (df["Weight_in_gms"] < lower_bound) | (df["Weight_in_gms"] > upper_bound)
outlier_count = outlier_mask.sum()
outlier_pct = outlier_mask.mean() * 100

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outlier count:", outlier_count)
print(f"Outlier percentage: {outlier_pct:.2f}%")


### Output Interpretation

- `Weight_in_gms` için 1.5×IQR yöntemi **hiçbir outlier işaretlemedi** (`lower_bound` negatif, `max`
  değeri `upper_bound`'ın altında kalıyor). Bu, bu değişkenin değerlerinin nispeten dar bir aralıkta
  (1001-7846 gms) ve ağır kuyruklu olmadan dağıldığını gösterir — "outlier bulunamadı" sonucu da geçerli
  ve bilgilendirici bir EDA bulgusudur.

### Important Rules

- **Outlier'ları orijinal dataset'ten silmeyin.** Bu LAB yalnızca tespit ve raporlama yapar; veriyi
  değiştirmez.
- Bir outlier; gerçek ama alışılmadık bir gözlem (örn. gerçekten çok ağır bir ürün), bir ölçüm/kayıt
  hatası veya farklı bir alt popülasyondan gelen bir veri noktası olabilir — **sebep, veri olmadan
  bilinemez**.
- **Statistical Outlier ≠ Data Error.** IQR yöntemi, istatistiksel olarak "beklenenden uzak" noktaları
  bulur; bu noktaların hatalı olduğu anlamına gelmez.
- **1.5 × IQR, mutlak bir hata tespit yöntemi değildir.** Bu, yaygın kullanılan ama keyfi bir eşiktir
  (örn. bazı analizler 3×IQR kullanır); "doğru" cevabı garanti etmez.
- Aykırı değer analizi, sonuçların doğru yorumlanması için **domain knowledge** (örn. "bu ürün kategorisi
  için 8 kg normal mi?") gerektirir.

### Student Exercise

`Cost_of_the_Product` için IQR yöntemiyle outlier oranını hesaplayın.


In [ ]:
# Student Exercise:
# TODO: Repeat the IQR outlier method above for "Cost_of_the_Product".
# TODO: Compute lower_bound, upper_bound, outlier_count, and outlier_pct.
# TODO: Compare your result with the "Weight_in_gms" result above.

# q1 = df["Cost_of_the_Product"].quantile(0.25)
# q3 = df["Cost_of_the_Product"].quantile(0.75)
# iqr = q3 - q1
# ...


---
## Section 4 — Data Distributions & Skewness
**Süre: 15 dakika**

### Learning Objective

Bir değişkenin dağılım şeklini histogram ve skewness (çarpıklık) ile değerlendirmek.

### Concept Explanation

- **Probability Distribution Fundamentals**: Bir değişkenin alabileceği değerlerin ve bu değerlerin ne
  sıklıkla gözlendiğinin genel şeklidir.
- **Frequency Distribution / Histogram**: Sayısal bir değişkenin değer aralıklarını (bins) kaç gözlemin
  doldurduğunu gösteren grafiktir.
- **Symmetric Distribution**: Dağılımın sol ve sağ yarısı birbirine yaklaşık ayna görüntüsüdür; mean ve
  median birbirine yakındır.
- **Right-Skewed (Positively Skewed) Distribution**: Sağa doğru uzun bir kuyruk vardır; genellikle
  mean > median.
- **Left-Skewed (Negatively Skewed) Distribution**: Sola doğru uzun bir kuyruk vardır; genellikle
  mean < median.
- **Normal Distribution**: Simetrik, tek tepeli, çan eğrisi şeklindeki teorik bir dağılımdır. Gerçek
  dünya verileri **her zaman** normal dağılmaz — bunu varsaymadan önce her zaman kontrol etmek gerekir.
- **Skewness**: Bir dağılımın asimetrisini sayısal olarak özetleyen bir ölçüdür. `0`'a yakın değerler
  simetriye, pozitif değerler sağa çarpıklığa, negatif değerler sola çarpıklığa işaret eder.

### Hands-On


In [ ]:
import matplotlib.pyplot as plt

BAR_COLOR = "#2a78d6"
MEAN_COLOR = "#eb6834"
MEDIAN_COLOR = "#1baf7a"
GRID_COLOR = "#e1e0d9"


def style_axes(ax):
    # Shared minimal style for all charts in this notebook.
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["Cost_of_the_Product"], bins=30, color=BAR_COLOR)
ax.axvline(df["Cost_of_the_Product"].mean(), color=MEAN_COLOR, linestyle="--", label="Mean")
ax.axvline(df["Cost_of_the_Product"].median(), color=MEDIAN_COLOR, linestyle="--", label="Median")
ax.set_title("Cost_of_the_Product — Distribution")
ax.set_xlabel("Cost of the Product")
ax.set_ylabel("Number of Shipments")
ax.legend()
style_axes(ax)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["Weight_in_gms"], bins=30, color=BAR_COLOR)
ax.axvline(df["Weight_in_gms"].mean(), color=MEAN_COLOR, linestyle="--", label="Mean")
ax.axvline(df["Weight_in_gms"].median(), color=MEDIAN_COLOR, linestyle="--", label="Median")
ax.set_title("Weight_in_gms — Distribution")
ax.set_xlabel("Weight (gms)")
ax.set_ylabel("Number of Shipments")
ax.legend()
style_axes(ax)
plt.show()


In [ ]:
cost_skew = df["Cost_of_the_Product"].skew()
weight_skew = df["Weight_in_gms"].skew()

print(f"Cost_of_the_Product skewness: {cost_skew:.4f}")
print(f"Weight_in_gms skewness      : {weight_skew:.4f}")


### Interpretation

- Her iki değişken de **negatif (sola çarpık) ama 0'a yakın** bir skewness değeri gösterir — bu,
  dağılımların *yaklaşık simetrik, hafifçe sola çarpık* olduğu, aşırı bir asimetri bulunmadığı anlamına
  gelir.
- Bu, Section 2'deki gözlemle tutarlıdır: her iki değişkende de **mean < median** idi — negatif
  skewness, mean'in median'ın solunda (daha küçük) olmasıyla örtüşür.
- **Distribution Shape, modelleme kararlarını etkileyebilir**: Örneğin, güçlü şekilde çarpık bir
  değişken, bazı modelleme teknikleri için dönüştürülebilir (örn. log transform) — ancak bu dönüşümler
  bu LAB'ın kapsamı dışındadır (Module 4 — Data Preparation).
- **Önemli:** Buradaki her iki değişken de normal dağılıma sahip olmak **zorunda değildir** ve biz bunu
  varsaymadık — sadece gerçek şeklini `skew()` ile ölçtük.

### Teorik Demonstrasyon — Normal Distribution

Aşağıdaki histogram, **gerçek shipping verisi değildir**. NumPy ile üretilmiş, tamamen teorik ve sentetik
bir Normal Distribution örneğidir; yalnızca simetrik bir "çan eğrisi"nin neye benzediğini göstermek
içindir.


In [ ]:
rng = np.random.default_rng(seed=42)
synthetic_normal_sample = rng.normal(loc=0, scale=1, size=10000)

print("This is a SYNTHETIC sample, not real shipping data.")
print("Synthetic sample skewness:", pd.Series(synthetic_normal_sample).skew())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(synthetic_normal_sample, bins=40, color=BAR_COLOR)
ax.set_title("Theoretical Normal Distribution (synthetic data, NumPy)")
ax.set_xlabel("Value")
ax.set_ylabel("Frequency")
style_axes(ax)
plt.show()


### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Compute the skewness of "Discount_offered" and "Prior_purchases".
# TODO: For each, state whether it is right-skewed, left-skewed, or approximately symmetric.

# print(df["Discount_offered"].skew())
# print(df["Prior_purchases"].skew())


---
## ☕ Break
**Süre: 10 dakika**

Kısa bir ara verin. Dönüşte **Data Quality Assessment** bölümüyle devam edeceğiz.


---
## Section 5 — Data Quality Assessment
**Süre: 20 dakika**

### Learning Objective

Veri kümesindeki tamlık (completeness), tekrar (duplication) ve geçerlilik (validity) sorunlarını
belirlemek ve özet bir Data Quality Report oluşturmak.

### Concept Explanation

- **Data Completeness / Missing Values**: Bir sütunda kaç değerin eksik (`NaN`) olduğu.
- **Duplicate Records**: Birden fazla satırın tamamen veya kritik alanlarda (örn. `ID`) birbirinin aynısı
  olması.
- **Invalid Values**: Domain bilgisiyle çelişen değerler (örn. negatif bir ağırlık, 1-5 aralığının
  dışında bir `Customer_rating`).
- **Data Type Validation**: Bir sütunun beklenen veri tipinde (sayısal, metinsel, tarih vb.) olup
  olmadığının kontrolü.
- **Range Checks**: Sayısal bir sütunun değerlerinin mantıklı bir aralıkta olup olmadığının kontrolü.
- **Categorical Consistency**: Kategorik bir sütunun beklenmedik yazım farklılıkları (örn. `"Flight"`
  vs `"flight"`) içermediğinin kontrolü.
- **Data Quality Reporting**: Bulguları, karar vericilerin (örn. bir sonraki LAB'daki Data Preparation
  ekibinin) kolayca okuyabileceği özet bir tabloda sunmak.
- **Sampling Bias Fundamentals**: Bir veri kümesinin, incelemek istediğiniz gerçek popülasyonu adil
  şekilde temsil etmeyebileceği fikridir (örn. yalnızca belirli bir bölgeden, belirli bir zaman
  diliminden toplanmış veri).

### Hands-On


In [ ]:
# 1 & 2. Missing value count and percentage per column.
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(3)

print("Missing value count:")
print(missing_count)
print()
print("Missing value percentage (%):")
print(missing_pct)


In [ ]:
# 3 & 4. Duplicate row count and duplicate ID count.
duplicate_rows = df.duplicated().sum()
duplicate_ids = df["ID"].duplicated().sum()

print("Duplicate row count:", duplicate_rows)
print("Duplicate ID count :", duplicate_ids)


In [ ]:
# 5. Numerical range inspection against plausible domain expectations.
numeric_range_checks = df[numerical_columns].agg(["min", "max"]).T
numeric_range_checks


`Customer_rating` (1-5 arası beklenir), `Discount_offered` (0-100 arası bir yüzde olması beklenir) ve
`Customer_care_calls` gibi sütunların gerçek min/max değerlerini yukarıdaki tabloda kontrol edin —
beklenen aralığın dışında bir değer, bir **Invalid Value** şüphesi doğurur.


In [ ]:
# 6. Categorical value inspection — look for unexpected spellings or casing issues.
for col in categorical_columns:
    print(f"{col}: {sorted(df[col].unique())}")


### Unexpected Values — Observation

`Warehouse_block` sütununda `A, B, C, D, F` değerleri bulunur — **`E` bloğu yoktur**. Bu, bir veri hatası
değildir; muhtemelen işletmenin kendi depo isimlendirme standardının bir sonucudur (örn. `E` kodu başka
bir amaç için ayrılmış olabilir). Bunu bir **gözlem** olarak not ediyoruz; kesin bir açıklamamız yok.


### Data Quality Report


In [ ]:
data_quality_report = pd.DataFrame(
    {
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Missing Count": df.isna().sum().values,
        "Missing Percentage": (df.isna().mean() * 100).round(3).values,
        "Unique Values": [df[col].nunique() for col in df.columns],
    }
)
data_quality_report


### Important Educational Distinction

Bu bölümde yalnızca veri kalitesi sorunlarını **belirledik ve raporladık**. Eksik değer doldurma
(imputation), kategorik kodlama (encoding) veya ölçekleme (scaling) gibi veriyi **değiştiren** işlemler
bu LAB'ın kapsamında değildir — bu işlemler **Module 4 — Data Preparation**'da ele alınacaktır.

### Sampling Bias

Bu dataset, hangi coğrafyadan, hangi zaman diliminden veya hangi iş süreçlerinden toplandığına dair bir
belge sağlamaz. Bu bilgi eksikliği, dataset'in "tüm e-ticaret gönderileri" popülasyonunu ne kadar iyi
temsil ettiğini değerlendirmemizi zorlaştırır. Bir **sampling bias** olup olmadığını kesin olarak
**söyleyemeyiz** — ancak bunu değerlendirebilmek için şu bilgilere ihtiyacımız olurdu:

- Verinin toplandığı zaman aralığı (örn. yalnızca bir kampanya dönemi mi?)
- Verinin toplandığı coğrafi/organizasyonel kapsam (örn. tek bir depo zinciri mi?)
- Veri toplama yönteminin kendisi (örn. yalnızca tamamlanan siparişler mi dahil edilmiş?)

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Pick two categorical columns of your choice (other than the ones already inspected above).
# TODO: Check whether their values look consistent (no unexpected spelling/casing variants).
# TODO: Report the missing value percentage for "Gender" specifically.

# print(df["Gender"].unique())
# print(df["Gender"].isna().mean() * 100)


---
## Section 6 — Correlation & Group Comparisons
**Süre: 20 dakika**

### Learning Objective

Sayısal değişkenler arasındaki doğrusal ilişkiyi (correlation) ölçmek ve kategorik gruplar arasında
istatistiksel karşılaştırmalar yapmak; her iki analiz türünün sınırlılıklarını anlamak.

### Concept Explanation

- **Correlation Fundamentals**: İki sayısal değişkenin birlikte nasıl değiştiğini ölçer.
- **Pearson Correlation**: İki değişken arasındaki **doğrusal (linear)** ilişkinin yönünü ve gücünü,
  `-1` ile `+1` arasında bir katsayı ile özetler. `0`'a yakın değerler zayıf doğrusal ilişkiye işaret
  eder (ama doğrusal olmayan bir ilişki hâlâ var olabilir!).
- **Correlation Matrix**: Birden fazla değişken çiftinin Pearson correlation değerlerini aynı anda
  gösteren bir tablodur.
- **Correlation vs Causation**: İki değişken birlikte hareket ediyor diye, biri diğerine **neden**
  olduğu anlamına gelmez.
- **Confounding Variables**: İki değişken arasındaki ilişkiyi açıklayabilecek, ölçülmemiş üçüncü bir
  değişkendir.
- **Group-Based Statistical Comparisons / Aggregation-Based EDA**: Veriyi kategorik bir değişkene göre
  gruplandırıp, her grup için özet istatistikleri karşılaştırmaktır (LAB02'deki `groupby` bilginizin
  istatistiksel yorumlama ile birleştirilmiş hâli).
- **Interpretation Limitations**: Gözlemlenen bir fark veya ilişkinin, istatistiksel anlamlılık testi
  yapılmadan "kesin" olarak kabul edilemeyeceği.

### Mathematical Explanation

Pearson correlation katsayısı:

$$r_{xy} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2} \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$

### Hands-On


In [ ]:
correlation_columns = [
    "Cost_of_the_Product",
    "Weight_in_gms",
    "Customer_care_calls",
    "Prior_purchases",
    "Discount_offered",
]

# Verify all expected columns actually exist before computing the matrix.
missing_corr_columns = [col for col in correlation_columns if col not in df.columns]
assert not missing_corr_columns, f"Missing expected columns: {missing_corr_columns}"

correlation_matrix = df[correlation_columns].corr()
correlation_matrix.round(3)


### Output Interpretation — Correlation

- En güçlü ilişki `Discount_offered` ile `Weight_in_gms` arasında görülür (**negatif, orta düzeyde**,
  ~`-0.38`): daha ağır gönderilerde indirim oranı eğiliminde bir azalma gözlenir.
- `Customer_care_calls` ile `Cost_of_the_Product` arasında **pozitif, orta düzeyde** bir ilişki vardır
  (~`0.32`): daha maliyetli ürünler için biraz daha fazla müşteri hizmetleri araması yapılmış olabilir.
- Diğer tüm çiftler **zayıf** (`|r| < 0.2`) doğrusal ilişki gösterir.
- Bu dataset'teki hiçbir çift **güçlü** (örn. `|r| > 0.7`) bir doğrusal ilişki göstermemektedir.
- **Not:** `ID` sütunu bir observation identifier olduğu için bu matrise dahil edilmemiştir —
  anlamsal olarak bir ölçüm değildir.

### Group Comparisons


In [ ]:
# Re-confirm the target encoding before using it as a rate (1 = Late, 0 = On Time).
assert sorted(df["Reached.on.Time_Y.N"].unique()) == [0, 1]


def group_comparison(column: str) -> pd.DataFrame:
    return (
        df.groupby(column)
        .agg(
            observation_count=("ID", "size"),
            mean_cost=("Cost_of_the_Product", "mean"),
            median_cost=("Cost_of_the_Product", "median"),
            std_cost=("Cost_of_the_Product", "std"),
            late_delivery_rate=("Reached.on.Time_Y.N", "mean"),
        )
        .reset_index()
    )


In [ ]:
group_comparison("Warehouse_block")


In [ ]:
group_comparison("Mode_of_Shipment")


In [ ]:
group_comparison("Product_importance")


### Interpretation — Group Comparisons

- **`Warehouse_block`**: `late_delivery_rate` tüm bloklarda `~%58-60` aralığında kalır — pratikte önemli
  bir fark görünmüyor. Ayrıca `F` bloğunun gözlem sayısı (`~3666`) diğer bloklardan (`~1833`) **iki kat
  fazladır**; bu, grup büyüklüklerinin eşit olmadığını ve karşılaştırmanın bu farkı dikkate alması
  gerektiğini gösterir.
- **`Mode_of_Shipment`**: `late_delivery_rate`, `Flight` için en yüksek, `Road` için en düşüktür, ama
  aralarındaki fark yalnızca birkaç yüzde puanıdır (`~%59-60`) — küçük bir fark.
- **`Product_importance`**: Burada en dikkat çekici fark görülür — `high` önem düzeyindeki ürünlerin
  `late_delivery_rate`'i (`~%65`), `low` ve `medium` düzeylerinden (`~%59`) belirgin şekilde daha
  yüksektir. **Ancak** `high` grubunun gözlem sayısı (`~948`) diğer gruplardan (`~4750-5300`) çok daha
  **küçüktür** — küçük örneklem büyüklüğü, gözlemlenen farkın ne kadar güvenilir olduğunu etkileyebilir.

### Critical Rules

- Pearson Correlation yalnızca **doğrusal (linear)** ilişkiyi ölçer; doğrusal olmayan güçlü bir ilişki
  olsa bile düşük bir Pearson değeri görülebilir.
- **Correlation ≠ Causation.** Gözlemlenen bir ilişki, bir değişkenin diğerine neden olduğunu kanıtlamaz.
- Kategorik değişkenleri (örn. `Warehouse_block`) keyfî sayısal kodlara (`A=1, B=2, ...`) çevirip Pearson
  Correlation'a dahil etmeyin — bu, var olmayan bir sıralama/doğrusallık varsayımı yaratır.
- Güçlü bir correlation bulunması, bir feature'ın sonucu **doğrudan etkilediğini** kanıtlamaz —
  confounding variables (ölçülmemiş ortak nedenler) her zaman olası bir açıklamadır.
- Grup farklarını (örn. `Product_importance` farkı), istatistiksel anlamlılık testi yapılmadan kesin bir
  anlamlılık veya nedensellik kanıtı olarak **sunmayın** — bu LAB'da yalnızca gözlemlenen farkları
  raporluyoruz.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Run group_comparison("Gender") and inspect the result.
# TODO: Compare the two groups' late_delivery_rate and observation_count.
# TODO: State in a comment whether the difference (if any) looks practically meaningful.

# group_comparison("Gender")


---
## Section 7 — EDA Visualization with Matplotlib
**Süre: 15 dakika**

### Learning Objective

EDA'ya özgü dört temel grafik türünü (histogram, boxplot, bar chart, scatter plot) oluşturmak ve
yorumlamak.

### Concept Explanation

- **Histogram**: Sayısal bir değişkenin frekans dağılımını gösterir (Section 4'te zaten kullandık).
- **Boxplot**: Q1, median (Q2), Q3, IQR sınırları ve potansiyel outlier'ları tek bir grafikte özetler.
- **Bar Chart**: Kategorik bir değişkenin kategorilerini karşılaştırır.
- **Scatter Plot**: İki sayısal değişken arasındaki ilişkiyi (varsa) görsel olarak gösterir — Section
  6'daki correlation değerini görsel olarak tamamlar.
- **Subplots / Figure & Axes**: Birden fazla grafiği aynı `Figure` içinde, ayrı `Axes` alanlarında
  düzenlemeyi sağlar.

Bu LAB'da **Seaborn kullanılmaz**; mevcut Matplotlib bağımlılığı yeterlidir.

### Hands-On

#### Chart 1 — Product Cost Distribution (Histogram)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["Cost_of_the_Product"], bins=30, color=BAR_COLOR)
ax.set_title("Product Cost Distribution")
ax.set_xlabel("Cost of the Product")
ax.set_ylabel("Number of Shipments")
style_axes(ax)
plt.show()


#### Chart 2 — Product Weight Distribution (Boxplot)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot(
    df["Weight_in_gms"],
    orientation="horizontal",
    patch_artist=True,
    boxprops={"facecolor": BAR_COLOR, "alpha": 0.7},
    medianprops={"color": MEAN_COLOR, "linewidth": 2},
)
ax.set_title("Product Weight Distribution")
ax.set_xlabel("Weight (gms)")
ax.set_yticks([])
style_axes(ax)
plt.show()


#### Chart 3 — Shipment Count by Transport Mode (Bar Chart)


In [ ]:
shipment_counts = df["Mode_of_Shipment"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(shipment_counts.index, shipment_counts.values, color=BAR_COLOR)
ax.set_title("Shipment Count by Transport Mode")
ax.set_xlabel("Mode of Shipment")
ax.set_ylabel("Number of Shipments")
style_axes(ax)
plt.show()


#### Chart 4 — Product Cost vs Weight (Scatter Plot)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["Weight_in_gms"], df["Cost_of_the_Product"], color=BAR_COLOR, alpha=0.2, s=10)
ax.set_title("Product Cost vs Weight")
ax.set_xlabel("Weight (gms)")
ax.set_ylabel("Cost of the Product")
style_axes(ax)
plt.show()


### Output Interpretation

- **Chart 1**, Section 4'teki histogramla aynı bilgiyi gösterir, ancak burada mean/median çizgileri
  yoktur — yalnızca genel şekli incelemek içindir.
- **Chart 2**, kutunun (box) nispeten dar olduğunu ve her iki yönde de "whisker" dışında nokta
  (outlier) bulunmadığını gösterir — bu, Section 3'teki "`Weight_in_gms` için outlier bulunamadı"
  bulgusuyla **tutarlıdır**.
- **Chart 3**, `Ship` modunun en sık kullanılan taşıma yöntemi olduğunu gösterir.
- **Chart 4**'teki noktalar, belirgin bir doğrusal örüntü oluşturmaz — bu, Section 6'da hesapladığımız
  zayıf negatif correlation (`~-0.13`) ile **görsel olarak tutarlıdır**. Grafikteki saçılma, iki
  değişken arasında güçlü bir doğrusal ilişki olmadığını teyit eder.

> Bu yorumlar gözlemsel niteliktedir; grafiklerden kesin sonuçlar (örn. "maliyet asla ağırlığa bağlı
> değildir") çıkarılmamalıdır.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: Create a boxplot for "Cost_of_the_Product" (reuse the Chart 2 code as a starting point).
# TODO: Compare its shape (box width, whisker length) with the "Weight_in_gms" boxplot above.

# fig, ax = plt.subplots(figsize=(6, 4))
# ax.boxplot(df["Cost_of_the_Product"], orientation="horizontal", ...)


---
## Section 8 — EDA Workshop & Findings
**Süre: 15 dakika**

### Workshop Objective

Bu bölümde, öğrendiğiniz istatistiksel yöntemleri kullanarak **kendi başınıza** kısa bir EDA
değerlendirmesi gerçekleştireceksiniz.

### Student Tasks

E-Commerce Shipping dataset'i üzerinde:

1. Dataset'in temel yapısını özetleyin (shape, kolon sayısı, hedef değişken).
2. En az üç numerical feature için descriptive statistics hesaplayın.
3. Bir değişkenin dağılımını inceleyin (histogram + skewness).
4. IQR ile potansiyel outlier'ları belirleyin (bu LAB'da henüz incelenmemiş bir değişken seçin, örn.
   `Discount_offered` veya `Customer_care_calls`).
5. En az iki kategorik grup arasında istatistiksel karşılaştırma yapın.
6. Bir correlation bulgusunu yorumlayın.
7. En az iki Data Quality bulgusu veya kontrol sonucu raporlayın.
8. Sonuçlarınızın sınırlamalarını (örn. küçük örneklem, nedensellik iddiası yapmama) açıklayın.


In [ ]:
# EDA Workshop — Scaffold
# Fill in each TODO using the techniques from Sections 1-7. This cell is intentionally
# left as a template so it does not block sequential notebook execution.

# TODO 1. Dataset overview
# print(df.shape)

# TODO 2. Descriptive statistics for >= 3 numerical features
# df[["...", "...", "..."]].describe()

# TODO 3. Distribution of one variable (histogram + skew())
# df["..."].skew()

# TODO 4. IQR outliers for a not-yet-explored variable
# q1 = df["..."].quantile(0.25)
# q3 = df["..."].quantile(0.75)

# TODO 5. Group comparison for >= 2 categorical groups
# group_comparison("...")

# TODO 6. Interpret one correlation value from correlation_matrix

# TODO 7. Report >= 2 data quality findings

# TODO 8. State the limitations of your findings in a comment


---
## EDA Findings — Instructor Summary

Bu bölüm, LAB03 boyunca gerçekleştirilen hesaplamalara dayanan, gerçek bulgulara dayalı bir özet
sunar. Aşağıdaki tüm ifadeler, yukarıdaki hücrelerde hesaplanan sonuçlardan alınmıştır.

### Dataset Overview

- 10999 gönderi (satır), 12 sütun. Eksik değer ve tam satır/ID duplikasyonu bulunmamaktadır.
- Hedef değişken (`Reached.on.Time_Y.N`): gönderilerin `~%60`'ı geç (`1 = Late`) olarak işaretlenmiştir.

### Statistical Findings

- `Cost_of_the_Product` ve `Weight_in_gms` için mean, median'den küçüktür (hafif sola çarpıklık).
- `Prior_purchases` için mean (~3.57), median'den (3.0) büyüktür — yüksek değerli birkaç müşteri
  ortalamayı yukarı çekmektedir (sağa çarpıklık, `skew ≈ 1.68`).
- `Customer_rating` oldukça simetrik dağılmıştır (mean ≈ median ≈ mode ≈ 3).

### Distribution Findings

- `Cost_of_the_Product` ve `Weight_in_gms`, 1.5×IQR yöntemine göre **outlier içermemektedir** — her iki
  değişken de dar ve kuyruksuz bir aralıkta dağılmıştır.
- `Prior_purchases`, aynı yöntemle **anlamlı oranda** potansiyel outlier içermektedir — bu, sağa çarpık
  dağılımıyla tutarlıdır.

### Data Quality Observations

- Eksik değer: **0** (tüm sütunlarda %0 missing).
- Duplicate satır ve duplicate `ID`: **0**.
- `Warehouse_block` sütununda `E` bloğu bulunmaz (`A, B, C, D, F`) — bir hata değil, muhtemelen bir
  işletme/isimlendirme kararı; kesin neden bilinmemektedir.

### Correlation Observations

- İncelenen 5 sayısal değişken arasında **güçlü** (|r| > 0.7) bir doğrusal ilişki bulunmamaktadır.
- En güçlü ilişkiler: `Discount_offered` ↔ `Weight_in_gms` (orta düzey negatif, ~-0.38) ve
  `Customer_care_calls` ↔ `Cost_of_the_Product` (orta düzey pozitif, ~0.32).

### Potential Risks

- `Product_importance` bazında gözlemlenen late-delivery-rate farkı (yüksek önem düzeyinde daha yüksek
  gecikme oranı), **küçük bir örneklem büyüklüğüne** (`~948` gözlem) dayanmaktadır — bu farkın
  güvenilirliği ayrıca test edilmeden kesinleştirilmemelidir.
- Dataset'in coğrafi/zamansal kapsamı belgelenmemiştir; bu, olası bir **sampling bias**'ı
  değerlendirmemizi zorlaştırır.
- Hiçbir bulgu, nedensellik (causation) kanıtı olarak sunulmamıştır — yalnızca gözlemlenen ilişkiler ve
  farklar raporlanmıştır.

### Recommendations for Data Preparation (Module 4 için notlar)

- `Prior_purchases` gibi sağa çarpık değişkenler için bir dönüşüm (örn. log transform) **Module 4**'te
  değerlendirilebilir.
- Eksik değer veya duplicate bulunmadığı için bu dataset için agresif bir temizleme adımı gerekli
  görünmemektedir — ancak bu, her yeni veri yüklemesinde yeniden kontrol edilmelidir.
- `Product_importance` gibi düşük gözlem sayılı kategoriler, modelleme öncesinde grup büyüklüğü
  açısından ayrıca değerlendirilmelidir.


---
## Advanced Challenges (opsiyonel — ana 150 dakikalık süreye dahil değildir)

Aşağıdaki üç challenge, deneyimli katılımcılar için hazırlanmıştır. Tamamlanmamış olmaları, notebook'un
sıralı çalışmasını etkilemez (tüm hücreler yalnızca yorum satırı içerir).

### Challenge 1 — Bootstrap Confidence Interval

Bir numerical feature için (örn. `Cost_of_the_Product`), **bootstrap** yöntemiyle mean için %95 güven
aralığı (confidence interval) hesaplayın: veriden `n` boyutunda, yerine koyarak (with replacement) tekrar
tekrar örnek alıp (resample), her örneğin mean'ini kaydedin; sonuçların 2.5. ve 97.5. percentile'ları
güven aralığının sınırlarını verir.


In [ ]:
# Advanced Challenge 1 — Bootstrap Confidence Interval (optional):
# TODO: Pick a numerical column, e.g. "Cost_of_the_Product".
# TODO: Resample it (with replacement) many times (e.g. 2000 times) using np.random.default_rng().choice.
# TODO: Compute the mean of each resample and collect the results into an array.
# TODO: Report the 2.5th and 97.5th percentiles of that array as the 95% confidence interval.

# rng = np.random.default_rng(seed=42)
# data = df["Cost_of_the_Product"].to_numpy()
# bootstrap_means = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(2000)]
# ci_lower, ci_upper = np.percentile(bootstrap_means, [2.5, 97.5])


### Challenge 2 — Advanced Group Comparison

`Warehouse_block` ve `Mode_of_Shipment` kombinasyonlarına göre çoklu grup analizi gerçekleştirin
(`groupby` ile iki sütunu birlikte kullanın — LAB02'deki multi-column grouping bilginizi hatırlayın).


In [ ]:
# Advanced Challenge 2 — Advanced Group Comparison (optional):
# TODO: Group by ["Warehouse_block", "Mode_of_Shipment"] together.
# TODO: Compute observation_count, mean_cost, and late_delivery_rate for each combination.
# TODO: Identify any combination with a notably small observation_count, and flag it as low-confidence.

# combo_summary = (
#     df.groupby(["Warehouse_block", "Mode_of_Shipment"])
#     .agg(observation_count=("ID", "size"), mean_cost=("Cost_of_the_Product", "mean"),
#          late_delivery_rate=("Reached.on.Time_Y.N", "mean"))
#     .reset_index()
# )


### Challenge 3 — Statistical Interpretation (Discussion, No Code Required)

Section 6'da hesapladığınız en güçlü correlation değerini (`Discount_offered` ↔ `Weight_in_gms`, `~-0.38`)
alın ve kısaca tartışın:

- Bu korelasyonun **neden** nedensellik kanıtı olmadığını açıklayın.
- Bu ilişkiyi açıklayabilecek, ölçülmemiş olası bir **confounding variable** önerin.
- Bu dataset'in hangi sınırlamaları (örn. eksik bağlamsal bilgi), bu ilişkiyi daha kesin yorumlamanızı
  engelliyor?

*(Bu challenge'ı, bir sonraki hücreye kendi yorumunuzu yazarak veya sözlü olarak tartışarak
tamamlayabilirsiniz.)*


---
## LAB03 Özeti

Bu LAB'da, LAB02'de öğrendiğiniz Pandas/NumPy becerilerini kullanarak:

- Descriptive statistics (mean, median, mode) hesapladık ve ne zaman farklı davrandıklarını gördük.
- Dispersion ölçülerini (variance, std, IQR) hesapladık ve IQR tabanlı outlier tespiti uyguladık.
- Histogram ve skewness ile dağılım şekillerini değerlendirdik.
- Sistematik bir Data Quality Report oluşturduk.
- Correlation matrix ve grup karşılaştırmalarıyla ilişkileri — nedensellik iddiası olmadan — inceledik.
- Matplotlib ile EDA'ya özgü dört temel grafik türünü oluşturduk.

Bir sonraki modülde (**Module 4 — Data Preparation**), burada belirlediğimiz bulguları (çarpık
dağılımlar, küçük gruplar, kategorik değişkenler) bir model eğitimine hazırlamak için nasıl
işleyeceğimizi öğreneceğiz.

Bir sonraki oturumda görüşmek üzere!
